# Fake news detection: ML part
TF-IDF / Bag of Words with Logistic Regression, Naive Bayes and Random Forest on the ISOT dataset.
Then a test on LIAR, word-level explanations, and saving the best model for the web app.

In [ ]:
import os
import sys
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

sys.path.append("../code")
from preprocess import clean_text

os.makedirs("../capstone/assets", exist_ok=True)
os.makedirs("../code/models", exist_ok=True)

## Data (0 = real, 1 = fake)

In [ ]:
fake = pd.read_csv("../code/data/Fake.csv")
true = pd.read_csv("../code/data/True.csv")
fake["label"] = 1
true["label"] = 0

df = pd.concat([fake, true], ignore_index=True)
df["raw"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()
df = df[df["raw"].str.len() > 0].drop_duplicates(subset="raw").reset_index(drop=True)

print(df["label"].value_counts().rename({0: "real", 1: "fake"}))

## Preprocessing

In [ ]:
df["clean"] = df["raw"].apply(clean_text)
df[["raw", "clean"]].head(3)

## Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.2, stratify=df["label"], random_state=42
)
print(len(X_train), "train,", len(X_test), "test")

## Train 2 vectorizers x 3 models
Random Forest takes a few minutes.

In [ ]:
vectorizers = {
    "TF-IDF": TfidfVectorizer(max_features=10000, ngram_range=(1, 2)),
    "BoW": CountVectorizer(max_features=10000, ngram_range=(1, 2)),
}
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
}

results, fitted = [], {}
for vname, vec in vectorizers.items():
    Xtr = vec.fit_transform(X_train)
    Xte = vec.transform(X_test)
    for mname, m in models.items():
        clf = clone(m).fit(Xtr, y_train)
        pred = clf.predict(Xte)
        results.append({
            "Vectorizer": vname, "Model": mname,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred),
            "Recall": recall_score(y_test, pred),
            "F1": f1_score(y_test, pred),
        })
        fitted[(vname, mname)] = (vec, clf, pred)
        print("done:", vname, "+", mname)

res = pd.DataFrame(results).sort_values("F1", ascending=False).reset_index(drop=True)
res.to_csv("../capstone/assets/results.csv", index=False)
res.round(4)

## Charts

In [ ]:
pivot = res.pivot(index="Model", columns="Vectorizer", values="F1")
ax = pivot.plot(kind="bar", figsize=(8, 5), rot=0)
ax.set_ylim(max(0, res["F1"].min() - 0.02), 1.0)
ax.set_ylabel("F1 score")
ax.set_title("F1 by model and vectorizer")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout()
plt.savefig("../capstone/assets/f1_comparison.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, ((vname, mname), (_, _, pred)) in zip(axes.ravel(), fitted.items()):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"])
    ax.set_title(f"{vname} + {mname}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("../capstone/assets/confusion_matrices.png", dpi=150)
plt.show()

## Test on a different dataset (LIAR)
Same models, no retraining. Needs LIAR's `train.tsv` in `data/`, otherwise this part is skipped.
fake = pants-fire / false / barely-true, real = true / mostly-true, half-true is dropped.

In [ ]:
LIAR_PATH = "../code/data/train.tsv"
FAKE_LABELS = {"pants-fire", "false", "barely-true"}
REAL_LABELS = {"true", "mostly-true"}

if os.path.exists(LIAR_PATH):
    liar = pd.read_csv(LIAR_PATH, sep="\t", header=None, quoting=3, usecols=[1, 2])
    liar.columns = ["liar_label", "statement"]
    liar = liar[liar["liar_label"].isin(FAKE_LABELS | REAL_LABELS)].copy()
    liar["label"] = liar["liar_label"].isin(FAKE_LABELS).astype(int)
    liar["clean"] = liar["statement"].apply(clean_text)
    liar = liar[liar["clean"].str.len() > 0]
    baseline = max(liar["label"].mean(), 1 - liar["label"].mean())
    print(f"{len(liar)} LIAR rows, majority class baseline = {baseline:.1%}")

    ext_rows = []
    for (vname, mname), (vec, clf, _) in fitted.items():
        pred = clf.predict(vec.transform(liar["clean"]))
        ext_rows.append({"Vectorizer": vname, "Model": mname,
                         "Accuracy": accuracy_score(liar["label"], pred),
                         "F1": f1_score(liar["label"], pred)})
    ext = pd.DataFrame(ext_rows)
    ext.to_csv("../capstone/assets/external_results.csv", index=False)

    gap = res.merge(ext, on=["Vectorizer", "Model"], suffixes=(" (ISOT)", " (LIAR)"))
    gap["Combination"] = gap["Vectorizer"] + " + " + gap["Model"]
    ax = gap.set_index("Combination")[["F1 (ISOT)", "F1 (LIAR)"]].plot(kind="barh", figsize=(9, 5))
    ax.set_xlabel("F1 score")
    ax.set_title("Trained on ISOT: tested on ISOT vs LIAR")
    plt.tight_layout()
    plt.savefig("../capstone/assets/generalization_gap.png", dpi=150)
    plt.show()
    print(gap[["Combination", "Accuracy (ISOT)", "Accuracy (LIAR)", "F1 (ISOT)", "F1 (LIAR)"]].round(4).to_string(index=False))
else:
    print(LIAR_PATH, "not found, skipping")

## Save the best model
Takes the best F1 whose files are under 90 MB (GitHub's limit is 100 MB).

In [ ]:
MAX_MB = 90
chosen = None
for _, row in res.iterrows():
    vec, clf, _ = fitted[(row["Vectorizer"], row["Model"])]
    joblib.dump(vec, "../code/models/vectorizer.joblib", compress=3)
    joblib.dump(clf, "../code/models/model.joblib", compress=3)
    size_mb = (os.path.getsize("../code/models/vectorizer.joblib") + os.path.getsize("../code/models/model.joblib")) / 1e6
    if size_mb <= MAX_MB:
        chosen = row
        break
    print(f"skipped {row['Vectorizer']} + {row['Model']} ({size_mb:.0f} MB)")

print(f"saved {chosen['Vectorizer']} + {chosen['Model']}, F1 = {chosen['F1']:.4f}, {size_mb:.1f} MB")

## Which words matter?
Random Forest and Naive Bayes are hard to read, so a Logistic Regression is trained on the same features
just for explanations. Positive coefficient = pushes toward fake, negative = toward real.

In [ ]:
explainer = LogisticRegression(max_iter=1000).fit(vec.transform(X_train), y_train)
joblib.dump(explainer, "../code/models/explainer.joblib", compress=3)

names = vec.get_feature_names_out()
coef = explainer.coef_[0]
order = coef.argsort()
idx = list(order[:15]) + list(order[-15:])
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh([names[i] for i in idx], coef[idx], color=["tab:green"] * 15 + ["tab:red"] * 15)
ax.set_title("Words pushing toward real (green) and fake (red)")
ax.set_xlabel("coefficient")
plt.tight_layout()
plt.savefig("../capstone/assets/top_words.png", dpi=150)
plt.show()

## Try it

In [ ]:
from ml_predict import explain_style, predict_style

samples = [
    "The U.S. Senate passed a bill on Tuesday to fund the government through December, officials said.",
    "BREAKING!!! Secret documents PROVE the elites are hiding the truth and the media will never tell you!",
    "NASA confirms aliens living under Antarctic ice, whistleblower says government covered it up for decades.",
    "The central bank held interest rates steady on Wednesday, citing persistent inflation.",
    "Scientists say drinking lemon water cures all cancers and doctors are hiding it from you.",
]
for s in samples:
    print(predict_style(s), "<-", s[:70])

e = explain_style(samples[1])
print("\nsample 2, fake-leaning:", [(x["word"], x["weight"]) for x in e["top_fake"]] if e else None)
print("sample 2, real-leaning:", [(x["word"], x["weight"]) for x in e["top_real"]] if e else None)